# ROP Detection Training

This notebook trains a ResNet-50 model to detect Retinopathy of Prematurity (ROP) in retinal images.

## Setup

First, let's install the required dependencies and set up our environment.

In [16]:
!pip install -q torch torchvision opencv-python-headless scikit-learn tqdm

In [17]:
import os
import sys
import glob
import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from tqdm import tqdm

# Check GPU availability
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cpu


## Dataset Preparation

Let's prepare our dataset for training.

In [18]:
# Define dataset path (Kaggle mounts datasets to /kaggle/input/)
DATASET_PATH = "/kaggle/input/datasets/jananowakova/retinal-image-dataset-of-infants-and-rop"

print("Checking dataset path...")
if os.path.exists(DATASET_PATH):
    print(f"Dataset found at {DATASET_PATH}")
else:
    print(f"Dataset not found at {DATASET_PATH}")
    print("Please ensure the dataset is properly attached to this notebook.")

Checking dataset path...
Dataset found at /kaggle/input/datasets/jananowakova/retinal-image-dataset-of-infants-and-rop


In [ ]:
def load_dataset(dataset_dir, sample_size=None):
    """
    Load and parse the ROP dataset from directory structure
    """
    print("Finding dataset files...")
    image_paths = glob.glob(os.path.join(dataset_dir, "**/*.jpg"), recursive=True)
    if len(image_paths) == 0:
        raise ValueError(f"No images found in {dataset_dir}")
    print(f"Found {len(image_paths)} images.")

    # The dataset stores every photo 3 times (images / images_stack / images_stack_without_captions).
    # Keep one copy per filename, preferring the caption-free version, so the same photo
    # can never end up in both train and test.
    image_paths.sort(key=lambda p: (0 if "without_captions" in p else 1, p))
    unique_paths = {}
    for path in image_paths:
        unique_paths.setdefault(os.path.basename(path), path)
    image_paths = sorted(unique_paths.values())
    print(f"After removing duplicate copies: {len(image_paths)} images.")

    if sample_size and sample_size < len(image_paths):
        print(f"Sampling {sample_size} images from dataset")
        import random
        image_paths = random.sample(image_paths, sample_size)
    labels = []
    valid_paths = []
    for path in image_paths:
        basename = os.path.basename(path)
        parts = basename.split("_")
        dg_part = [p for p in parts if p.startswith("DG")]
        if dg_part:
            try:
                dg_code = int(dg_part[0].replace("DG", ""))
                # Codes from Table 10 of the dataset paper (Timkovic et al., Sci Data 2024):
                # 0 = physiological (normal); 2/3/4 = ROP stage 1/2/3; 8 = aggressive ROP;
                # 9 = treated ROP. Skipped: 1 (stage 0, borderline), 10-13 (other diseases,
                # e.g. 11 = retinal hemorrhage), which are neither normal nor ROP.
                if dg_code == 0:
                    labels.append(0)
                    valid_paths.append(path)
                elif dg_code in [2, 3, 4, 8, 9]:
                    labels.append(1)
                    valid_paths.append(path)
            except ValueError:
                print(f"Warning: Could not parse DG code from {basename}")
    print(f"Filtered to {len(valid_paths)} images with clear Normal/ROP labels.")
    return valid_paths, labels

In [ ]:
# Load dataset (using the full dataset for better performance)
image_paths, labels = load_dataset(DATASET_PATH)  # No sampling

## Clinic Data

Our own clinic photos. Half of the clinic **babies** are used for training, the other half only for the final test (never the same baby on both sides).

Attach the clinic dataset (Add Input → Your Datasets) before running.

In [ ]:
import hashlib
import random

# Our own clinic photos: folder name -> true label (1 = ROP, 0 = normal)
CLINIC_FOLDERS = {
    "plus pre": 1,
    "stage 3 neovascularization with ridge": 1,
    "treated ROP": 1,
    "normal vascularized retina": 0,
}
CLINIC_TRAIN_FRACTION = 0.5   # share of clinic babies used for training; the rest are only for the final test
CLINIC_REPEAT = 5             # clinic training photos are repeated so they aren't drowned out by Kaggle photos

def find_folder(name):
    """Locate a clinic folder anywhere under /kaggle/input (mount path varies)."""
    hits = [h for h in glob.glob(f"/kaggle/input/**/{glob.escape(name)}", recursive=True) if os.path.isdir(h)]
    return hits[0] if hits else None

def clinic_baby(path):
    """
    Clinic filenames look like L_43_127_2025-06-10_....jpg, so the baby id is the 3rd part.
    Prefixed with "clinic_" so clinic ids never clash with Kaggle ids.
    """
    return "clinic_" + os.path.basename(path).split("_")[2]

# Collect clinic photos, skipping exact duplicates (same photo saved twice or in two folders)
seen_hashes = set()
clinic_rows = []   # (folder, path, label)
for folder, label in CLINIC_FOLDERS.items():
    folder_path = find_folder(folder)
    if folder_path is None:
        print(f"Missing clinic folder: {folder} (is the clinic dataset attached?)")
        continue
    for path in sorted(glob.glob(os.path.join(folder_path, "*.jpg"))):
        with open(path, "rb") as f:
            digest = hashlib.md5(f.read()).hexdigest()
        if digest in seen_hashes:
            continue
        seen_hashes.add(digest)
        clinic_rows.append((folder, path, label))

# Split clinic BABIES, not photos. Babies with any ROP photo and normal-only babies are
# divided separately so both sides get a fair share. Fixed seed -> same split every run.
clinic_rop_babies = sorted({clinic_baby(p) for _, p, l in clinic_rows if l == 1})
clinic_other_babies = sorted({clinic_baby(p) for _, p, _ in clinic_rows} - set(clinic_rop_babies))
rng = random.Random(42)
clinic_train_babies = set()
for group in (clinic_rop_babies, clinic_other_babies):
    group = group[:]
    rng.shuffle(group)
    clinic_train_babies.update(group[:round(len(group) * CLINIC_TRAIN_FRACTION)])

clinic_train_rows = [r for r in clinic_rows if clinic_baby(r[1]) in clinic_train_babies]
clinic_test_rows = [r for r in clinic_rows if clinic_baby(r[1]) not in clinic_train_babies]

for name, rows_ in (("Clinic train", clinic_train_rows), ("Clinic test", clinic_test_rows)):
    babies = {clinic_baby(p) for _, p, _ in rows_}
    rop = sum(l for _, _, l in rows_)
    print(f"{name:<12} photos: {len(rows_):3d} | ROP photos: {rop:3d} | normal photos: {len(rows_) - rop:3d} | babies: {len(babies)}")

## Data Preprocessing

Define our custom dataset class with preprocessing.

In [ ]:
import random

# Image size fed to the model. 448 keeps the thin ridge line of early (stage 1-2) ROP visible;
# at 224 much of that detail is lost.
IMG_SIZE = 448

class ROPDataset(Dataset):
    def __init__(self, image_paths, labels, transform=None, augment=False):
        self.image_paths = image_paths
        self.labels = labels
        self.transform = transform
        self.augment = augment

    def __len__(self):
        return len(self.image_paths)

    def camera_jitter(self, img):
        """
        Random brightness, contrast and color-cast changes on the raw photo (training only).
        Simulates different cameras/lighting so the model copes better with clinic images.
        """
        # Python's `random` (not np.random) so each DataLoader worker gets different values
        contrast = random.uniform(0.8, 1.2)
        brightness = random.uniform(-20, 20)
        color_gain = np.array([random.uniform(0.9, 1.1) for _ in range(3)])
        img = img.astype(np.float32) * contrast * color_gain + brightness
        return np.clip(img, 0, 255).astype(np.uint8)

    def preprocess_image(self, img):
        """
        Apply preprocessing inspired by the Nature article: ReLeSeT tool
        """
        # 1. Convert to RGB (OpenCV loads in BGR)
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        # 2. Extract Red and Green channels, zero out Blue for noise reduction
        r, g, b = cv2.split(img_rgb)
        b = np.zeros_like(b)
        img_rg = cv2.merge([r, g, b])

        # 3. Bilateral Filtering for edge-preserving smoothing
        img_filtered = cv2.bilateralFilter(img_rg, d=9, sigmaColor=3, sigmaSpace=0.1)

        # 4. CLAHE (Adaptive Histogram Equalization) on the luminance/intensity
        lab = cv2.cvtColor(img_filtered, cv2.COLOR_RGB2LAB)
        l, a, _b = cv2.split(lab)
        clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
        cl = clahe.apply(l)
        limg = cv2.merge((cl, a, _b))
        final_img = cv2.cvtColor(limg, cv2.COLOR_LAB2RGB)

        return final_img

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        label = self.labels[idx]

        # Load image
        img = cv2.imread(img_path)
        if img is None:
            # Fallback if image fails to load
            print(f"Warning: Could not load image {img_path}, using blank image")
            img = np.zeros((IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8)
        else:
            img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
            if self.augment:
                img = self.camera_jitter(img)
            img = self.preprocess_image(img)

        if self.transform:
            img = self.transform(img)

        return img, torch.tensor(label, dtype=torch.float32)

## Model Definition

Create our ResNet-50 model for ROP classification.

In [21]:
def create_model(device, pretrained=True):
    """
    Create and configure the ResNet-50 model for ROP classification
    """
    if pretrained:
        model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)
    else:
        model = models.resnet50()

    num_ftrs = model.fc.in_features
    model.fc = nn.Sequential(
        nn.Dropout(0.5),
        nn.Linear(num_ftrs, 1),
        nn.Sigmoid()
    )
    model = model.to(device)
    return model

# Create model
model = create_model(device)
print("Model created successfully!")

Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 152MB/s] 


Model created successfully!


## Training Setup

Prepare data loaders and training parameters.

In [ ]:
import random

# Fixed seeds so two runs of the notebook give comparable results
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

def create_data_loaders(X_train, X_val, X_test, y_train, y_val, y_test, batch_size, num_workers=0):
    """
    Create training, validation and test data loaders with appropriate transforms
    """
    # Transforms
    train_transform = transforms.Compose([
        transforms.ToPILImage(),
        transforms.RandomHorizontalFlip(),
        transforms.RandomAffine(degrees=10, scale=(0.9, 1.1)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])

    eval_transform = transforms.Compose([
        transforms.ToPILImage(),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])

    train_dataset = ROPDataset(X_train, y_train, transform=train_transform, augment=True)
    val_dataset = ROPDataset(X_val, y_val, transform=eval_transform)
    test_dataset = ROPDataset(X_test, y_test, transform=eval_transform)

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=num_workers)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers)

    return train_loader, val_loader, test_loader

def patient_id(path):
    if os.path.basename(os.path.dirname(path)) in CLINIC_FOLDERS:
        return clinic_baby(path)
    return os.path.basename(path).split("_")[0]

def split_by_patient(paths, labels, val_frac=0.15, test_frac=0.2, seed=42):
    """
    Split babies into train / validation / test. All photos of one baby stay in one group.
    ROP babies and non-ROP babies are shuffled and divided separately, so every group gets
    its fair share of the (few) ROP babies. Uses a fixed seed, so the split is identical
    on Kaggle and locally regardless of library versions.
    """
    all_babies = sorted({patient_id(p) for p in paths})
    rop_babies = sorted({patient_id(p) for p, l in zip(paths, labels) if l == 1})
    other_babies = [b for b in all_babies if b not in set(rop_babies)]

    rng = random.Random(seed)
    val_babies, test_babies = set(), set()
    for group in (rop_babies, other_babies):
        group = group[:]
        rng.shuffle(group)
        n_test = max(1, round(len(group) * test_frac))
        n_val = max(1, round(len(group) * val_frac))
        test_babies.update(group[:n_test])
        val_babies.update(group[n_test:n_test + n_val])

    splits = {"train": ([], []), "val": ([], []), "test": ([], [])}
    for path, label in zip(paths, labels):
        baby = patient_id(path)
        name = "test" if baby in test_babies else "val" if baby in val_babies else "train"
        splits[name][0].append(path)
        splits[name][1].append(label)
    return splits["train"], splits["val"], splits["test"]

def describe(name, paths, labels):
    babies = {patient_id(p) for p in paths}
    rop_babies = {patient_id(p) for p, l in zip(paths, labels) if l == 1}
    print(f"{name:<11} photos: {len(paths):5d} | ROP photos: {sum(labels):4d} | babies: {len(babies):3d} | ROP babies: {len(rop_babies)}")

# Three groups of babies:
#   train      - the model learns from these
#   validation - used only to pick the best epoch
#   test       - used once at the very end, so its score is honest
(X_train, y_train), (X_val, y_val), (X_test, y_test) = split_by_patient(image_paths, labels)

# Add the clinic training babies. Repeated CLINIC_REPEAT times (each copy gets different random
# augmentation) so ~100 clinic photos aren't drowned out by ~3,000 Kaggle photos.
X_train = X_train + [path for _, path, _ in clinic_train_rows] * CLINIC_REPEAT
y_train = y_train + [label for _, _, label in clinic_train_rows] * CLINIC_REPEAT

describe("Train", X_train, y_train)
describe("Validation", X_val, y_val)
describe("Test", X_test, y_test)

# Create data loaders
train_loader, val_loader, test_loader = create_data_loaders(
    X_train, X_val, X_test, y_train, y_val, y_test, batch_size=8, num_workers=2
)

## Training Loop

Train the model with progress tracking.

In [ ]:
from sklearn.metrics import balanced_accuracy_score, recall_score

def train_model(model, train_loader, val_loader, device, epochs, learning_rate, save_path="best_rop_model.pth"):
    """
    Train the ROP detection model. The best epoch is picked on the validation babies,
    so the test babies stay untouched until the final evaluation.
    """
    # ROP photos are the minority, so a missed ROP photo costs more than a false alarm.
    # Weight = number of normal photos / number of ROP photos in the training set.
    train_labels = np.array(train_loader.dataset.labels)
    rop_weight = float((train_labels == 0).sum() / max((train_labels == 1).sum(), 1))
    print(f"ROP weight in loss: {rop_weight:.2f}")
    criterion = nn.BCELoss(reduction="none")
    optimizer = optim.AdamW(model.parameters(), lr=learning_rate)
    # Learning rate slowly decreases to near zero, so later epochs make smaller, steadier changes
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    # Mixed precision: GPU does most math in 16-bit, which is much faster on Kaggle's T4.
    use_amp = device.type == "cuda"
    try:
        scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
    except AttributeError:  # older PyTorch
        scaler = torch.cuda.amp.GradScaler(enabled=use_amp)

    best_score = 0.0

    print(f"Starting training for {epochs} epochs...")

    for epoch in range(epochs):
        # Training phase
        model.train()
        running_loss = 0.0

        # Use tqdm for progress bar
        progress_bar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{epochs}')
        for inputs, targets in progress_bar:
            inputs, targets = inputs.to(device), targets.to(device)

            optimizer.zero_grad()
            with torch.autocast(device_type=device.type, enabled=use_amp):
                # view(-1) keeps a 1-D shape even when the last batch has a single image
                outputs = model(inputs).view(-1)
            sample_weights = torch.where(targets == 1, rop_weight, 1.0).to(device)
            # BCELoss must run in full precision, so convert outputs back to float32
            loss = (criterion(outputs.float(), targets) * sample_weights).mean()
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            running_loss += loss.item()
            progress_bar.set_postfix({'loss': f"{loss.item():.4f}"})

        current_lr = scheduler.get_last_lr()[0]
        scheduler.step()

        # Validation phase
        model.eval()
        all_preds = []
        all_targets = []

        with torch.no_grad():
            for inputs, targets in val_loader:
                inputs = inputs.to(device)
                outputs = model(inputs).view(-1)
                preds = (outputs > 0.5).cpu().numpy()
                all_preds.extend(preds)
                all_targets.extend(targets.numpy())

        acc = accuracy_score(all_targets, all_preds)
        rop_recall = recall_score(all_targets, all_preds, zero_division=0)
        # Balanced accuracy = average of (normal caught) and (ROP caught); can't be gamed by
        # always answering "normal" like plain accuracy can
        score = balanced_accuracy_score(all_targets, all_preds)
        avg_loss = running_loss / len(train_loader)
        print(f"Epoch {epoch+1} - LR: {current_lr:.2e} - Loss: {avg_loss:.4f} - Val Acc: {acc:.4f} - Val ROP recall: {rop_recall:.4f} - Val Balanced: {score:.4f}")

        # Save best model (by validation balanced accuracy)
        if score > best_score:
            best_score = score
            torch.save(model.state_dict(), save_path)
            print(f"New best model saved with validation balanced accuracy: {score:.4f}")

    print(f"Training completed. Best validation balanced accuracy: {best_score:.4f}")

    # Return the best epoch's weights, not the last epoch's (later epochs can overfit)
    model.load_state_dict(torch.load(save_path, map_location=device))
    print(f"Loaded best model from {save_path}")
    return model

# Train the model
trained_model = train_model(model, train_loader, val_loader, device, epochs=10, learning_rate=5e-5)

## Model Evaluation

Evaluate the trained model on the test set.

In [ ]:
from sklearn.metrics import roc_auc_score, roc_curve, confusion_matrix

def get_probabilities(model, loader, device, desc="Predicting"):
    """Return (ROP probabilities, true labels) for every photo in the loader."""
    model.eval()
    all_probs, all_targets = [], []
    with torch.no_grad():
        for inputs, targets in tqdm(loader, desc=desc):
            all_probs.extend(model(inputs.to(device)).view(-1).cpu().numpy())
            all_targets.extend(targets.numpy())
    return np.array(all_probs), np.array(all_targets).astype(int)

def choose_threshold(probs, targets):
    """
    Pick the cut-off with the best balance between catching ROP and not flagging normal eyes
    (highest sensitivity + specificity, also called Youden's J).
    Chosen on the VALIDATION babies only, so the test score stays honest.
    """
    fpr, tpr, thresholds = roc_curve(targets, probs)
    best = np.argmax(tpr - fpr)
    return float(min(thresholds[best], 1.0))

def report(name, probs, targets, threshold):
    preds = (probs >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(targets, preds, labels=[0, 1]).ravel()
    print(f"{name} @ threshold {threshold:.3f}: "
          f"Sensitivity (ROP caught) {tp}/{tp + fn} = {tp / max(tp + fn, 1):.1%} | "
          f"Specificity (normal called normal) {tn}/{tn + fp} = {tn / max(tn + fp, 1):.1%}")

# 1. Pick the cut-off on the validation babies
val_probs, val_targets = get_probabilities(trained_model, val_loader, device, desc="Validation")
ROP_THRESHOLD = choose_threshold(val_probs, val_targets)
print(f"\nChosen ROP threshold (from validation): {ROP_THRESHOLD:.3f}")

# 2. Evaluate once on the untouched test babies
test_probs, test_targets = get_probabilities(trained_model, test_loader, device, desc="Test")
print(f"\nTest ROC AUC: {roc_auc_score(test_targets, test_probs):.4f}\n")
report("Test", test_probs, test_targets, 0.5)
report("Test", test_probs, test_targets, ROP_THRESHOLD)

test_preds = (test_probs >= ROP_THRESHOLD).astype(int)
print(f"\nClassification Report (threshold {ROP_THRESHOLD:.3f}):")
print(classification_report(test_targets, test_preds, target_names=["Normal", "ROP"], zero_division=0))

final_accuracy = accuracy_score(test_targets, test_preds)
print(f"Balanced Accuracy: {balanced_accuracy_score(test_targets, test_preds):.4f}")

# Which kinds of photos does the model get wrong? (Kaggle test, by diagnosis code and by ROP baby)
DG_NAMES = {0: "Normal", 2: "ROP stage 1", 3: "ROP stage 2", 4: "ROP stage 3", 8: "Aggressive ROP", 9: "Treated ROP"}
dg_codes = [int([part for part in os.path.basename(x).split("_") if part.startswith("DG")][0][2:]) for x in X_test]
breakdown = pd.DataFrame({
    "diagnosis": [DG_NAMES.get(c, f"DG{c}") for c in dg_codes],
    "baby": [patient_id(x) for x in X_test],
    "rop_probability": test_probs,
    "correct": test_preds == test_targets,
})
summary = dict(correct=("correct", "sum"), total=("correct", "count"), babies=("baby", "nunique"), avg_rop_probability=("rop_probability", "mean"))
print(f"\nCorrect answers by diagnosis (threshold {ROP_THRESHOLD:.3f}):")
print(breakdown.groupby("diagnosis").agg(**summary).assign(rate=lambda d: d.correct / d.total))
print("\nROP babies in the test group:")
print(breakdown[breakdown.diagnosis != "Normal"].groupby(["baby", "diagnosis"]).agg(**summary).assign(rate=lambda d: d.correct / d.total))


## Save Model

Save the trained model for future use.

In [25]:
import json

# Save the best model and the settings the Streamlit app needs (threshold + image size)
torch.save(trained_model.state_dict(), "/kaggle/working/rop_detection_model.pth")
with open("/kaggle/working/rop_model_config.json", "w") as f:
    json.dump({"threshold": ROP_THRESHOLD, "img_size": IMG_SIZE}, f, indent=2)
print("Model saved to /kaggle/working/rop_detection_model.pth")
print(f"Settings saved to /kaggle/working/rop_model_config.json (threshold {ROP_THRESHOLD:.3f}, image size {IMG_SIZE})")

Model saved successfully to /kaggle/working/rop_detection_model.pth


## Clinic Test (held-out clinic babies)

Test the trained model on the clinic babies that were **not** used for training (see "Clinic Data").
- ROP: `plus pre`, `stage 3 neovascularization with ridge`, `treated ROP`
- Normal: `normal vascularized retina`

In [ ]:
external_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

print(f"Clinic test images: {len(clinic_test_rows)} "
      f"(ROP: {sum(r[2] for r in clinic_test_rows)}, normal: {sum(1 - r[2] for r in clinic_test_rows)})")

clinic_test_dataset = ROPDataset([r[1] for r in clinic_test_rows], [r[2] for r in clinic_test_rows], transform=external_transform)
clinic_test_loader = DataLoader(clinic_test_dataset, batch_size=8, shuffle=False)
probs, _ = get_probabilities(trained_model, clinic_test_loader, device, desc="Clinic test")

results = pd.DataFrame({
    "folder": [r[0] for r in clinic_test_rows],
    "file": [os.path.basename(r[1]) for r in clinic_test_rows],
    "baby": [clinic_baby(r[1]) for r in clinic_test_rows],
    "true_label": [r[2] for r in clinic_test_rows],
    "rop_probability": probs,
})
# Use the cut-off chosen on the validation babies (see Model Evaluation)
results["predicted_rop"] = (results["rop_probability"] >= ROP_THRESHOLD).astype(int)
results["correct"] = results["predicted_rop"] == results["true_label"]

print()
report("Clinic", results.rop_probability.values, results.true_label.values, 0.5)
report("Clinic", results.rop_probability.values, results.true_label.values, ROP_THRESHOLD)

print(f"\nCorrect answers per folder (threshold {ROP_THRESHOLD:.3f}):")
print(results.groupby("folder")["correct"].agg(correct="sum", total="count").assign(rate=lambda d: d.correct / d.total))

if results.true_label.nunique() == 2:
    print(f"\nClinic ROC AUC: {roc_auc_score(results.true_label, results.rop_probability):.4f}")
    print(f"\nClassification Report (threshold {ROP_THRESHOLD:.3f}):")
    print(classification_report(results.true_label, results.predicted_rop, target_names=["Normal", "ROP"], zero_division=0))

results.to_csv("/kaggle/working/clinic_test_results.csv", index=False)
print("Per-image results saved to /kaggle/working/clinic_test_results.csv")

## Exam-Level Results

Doctors judge an **exam** (all photos of one eye at one visit), not single photos. Mild ROP often sits at the edge of the retina, so some photos from an ROP exam don't show it, yet every photo carries the exam's label. Here an exam counts as "ROP" if **any** of its photos scores above the threshold.

This cell only uses results already computed above, so it can be run on its own after the Clinic Test.

In [ ]:
def exam_level(df, name):
    """Group photos into exams; an exam is ROP if any photo is ROP (true) / scores above the threshold (predicted)."""
    exams = df.groupby("exam").agg(
        diagnosis=("diagnosis", "first"),
        true_label=("true_label", "max"),
        max_prob=("rop_probability", "max"),
        photos=("rop_probability", "size"),
    )
    exams["predicted_rop"] = (exams.max_prob >= ROP_THRESHOLD).astype(int)
    exams["correct"] = exams.predicted_rop == exams.true_label
    print(f"\n=== {name}: {len(exams)} exams ({exams.true_label.sum()} ROP, {(exams.true_label == 0).sum()} normal), "
          f"median {exams.photos.median():.0f} photos per exam")
    report(f"{name} exams", exams.max_prob.values, exams.true_label.values, ROP_THRESHOLD)
    if exams.true_label.nunique() == 2:
        print(f"{name} exam AUC: {roc_auc_score(exams.true_label, exams.max_prob):.4f}")
    print(exams.groupby("diagnosis").agg(correct=("correct", "sum"), total=("correct", "count")).assign(rate=lambda d: d.correct / d.total))
    return exams

# Kaggle test: filename = ID_sex_GA_BW_PA_DG_PF_D_S_imageNumber.jpg -> exam = everything except the image number
kaggle_exams = exam_level(pd.DataFrame({
    "exam": ["_".join(os.path.basename(x).split("_")[:-1]) for x in X_test],
    "diagnosis": breakdown.diagnosis.values,
    "true_label": test_targets,
    "rop_probability": test_probs,
}), "Kaggle test")

# Clinic test: filename = eye_frame_baby_date_time.jpg -> exam = baby + eye (L/R) + date.
# An exam with photos in both the normal and an ROP folder is an ROP exam (normal-looking area of an ROP eye).
clinic_exam_df = results.assign(exam=[f"{b}_{f[0]}_{f.split('_')[3]}" for b, f in zip(results.baby, results.file)])
clinic_exam_df["diagnosis"] = clinic_exam_df.groupby("exam")["folder"].transform(
    lambda s: "normal only" if (s == "normal vascularized retina").all() else "has ROP photos")
clinic_exams = exam_level(clinic_exam_df, "Clinic test")


## Conclusion

We have successfully trained a ResNet-50 model to detect Retinopathy of Prematurity in retinal images. The model achieved an accuracy of {:.4f} on the test set.

The trained model has been saved and can be used for inference on new retinal images.

### Next Steps

1. Experiment with different architectures and hyperparameters
2. Increase the training dataset size for better generalization
3. Implement cross-validation for more robust evaluation
4. Try ensemble methods for improved performance
5. Deploy the model as an API for clinical use